### NBA MVP Prediction Model Comparison

Logistic Regression, Random Forest, XGBoost, and LightGBM models

In [1]:
# Imports

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt

In [2]:
# Load the dataset

train = pd.read_csv("Data/train.csv")
valid = pd.read_csv("Data/valid.csv")
test = pd.read_csv("Data/test.csv")

print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)

Train: (14679, 34)
Validation: (3196, 34)
Test: (4399, 34)


In [ ]:
# Choose features
TARGET = "Rk"

DROP_COLUMNS = [
    "Player",
    "Season",
    "Rk"
]

FEATURES = ["Age", "G", "MP", "PTS", "TRB", "AST", "STL", "BLK", "FG%", "3P%", "FT%", "TOV", "FTA"]

X_train = train[FEATURES]
y_train = train[TARGET]

X_valid = valid[FEATURES]
y_valid = valid[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print(len(FEATURES), "features")

11 features


In [51]:
# Le models

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=600, class_weight="balanced", random_state=2),
    "XGBoost": XGBClassifier(n_estimators=600, max_depth=11, learning_rate=0.01, scale_pos_weight=450, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=300, class_weight="balanced",random_state=42)
}

In [52]:
# Training functions

def evaluate_model(name, model):

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

NaN Handle

In [53]:
print(X_train.isna().sum().sort_values(ascending=False))

Age    0
G      0
MP     0
PTS    0
TRB    0
AST    0
STL    0
BLK    0
FG%    0
3P%    0
FT%    0
dtype: int64


In [54]:
# NaN usually means "no attempts"
percentage_cols = ["FG%", "3P%", "2P%", "FT%", "eFG%"]

for col in percentage_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna(0)
        X_valid[col] = X_valid[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)

# median everything else
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_valid = pd.DataFrame(imputer.transform(X_valid), columns=X_valid.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

In [55]:
# Train all the models

results = []

trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    trained_models[name] = model

    results.append(evaluate_model(name, model))

results = pd.DataFrame(results)

results

Training Logistic Regression...
Training Random Forest...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Number of positive: 14651, number of negative: 28
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000459 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1709
[LightGBM] [Info] Number of data points in the train set: 14679, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.964765,1.000000,0.964717,0.982041
1,Random Forest,0.998636,0.998636,1.000000,0.999318
2,XGBoost,0.998636,0.998636,1.000000,0.999318
3,LightGBM,0.998863,1.000000,0.998862,0.999431


In [56]:
# compare

results.sort_values(
    by="F1",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1
3,LightGBM,0.998863,1.000000,0.998862,0.999431
1,Random Forest,0.998636,0.998636,1.000000,0.999318
2,XGBoost,0.998636,0.998636,1.000000,0.999318
0,Logistic Regression,0.964765,1.000000,0.964717,0.982041


In [ ]:
# Feature importance

rf = trained_models["Random Forest"]

importance = pd.Series(
    rf.feature_importances_,
    index=FEATURES
)

importance = importance.sort_values()

plt.figure(figsize=(8,10))
importance.plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.show()

In [ ]:
# XGBoost feature importance

xgb = trained_models["XGBoost"]

importance = pd.Series(
    xgb.feature_importances_,
    index=FEATURES
)

importance = importance.sort_values()

plt.figure(figsize=(8,10))
importance.plot(kind="barh")
plt.title("XGBoost Feature Importance")
plt.show()

In [59]:
# Predict

best_model = trained_models["XGBoost"]

test_copy = test.copy()

test_copy["Probability"] = best_model.predict_proba(X_test)[:,0]

In [60]:
# MVP

for season in sorted(test_copy["season"].unique()):
    print(f"\n{season}")

    display(
        test_copy[test_copy["season"] == season]
        .sort_values("Probability", ascending=False)
        [["Player","Probability","Rk"]]
        .head(10)
    )


20-21


,Player,Probability,Rk
8,Kevin Durant,0.012132,1
10,Nikola Jokić,0.004560,0
4,Giannis Antetokounmpo,0.004391,1
9,Kyrie Irving,0.002563,1
16,LeBron James,0.002453,1
33,Russell Westbrook,0.002222,1
6,Zach LaVine,0.001850,1
17,Kawhi Leonard,0.001146,1
22,James Harden,0.000914,1
20,James Harden,0.000914,1



21-22


,Player,Probability,Rk
706,LeBron James,0.007274,1
714,Nikola Jokić,0.006271,0
708,Kevin Durant,0.005744,1
707,Giannis Antetokounmpo,0.004391,1
709,Luka Dončić,0.001034,1
710,Trae Young,0.000993,1
711,DeMar DeRozan,0.000936,1
734,James Harden,0.000707,1
719,Karl-Anthony Towns,0.000471,1
705,Joel Embiid,0.000469,1



22-23


...

,Player,Probability,Rk
2199,Shai Gilgeous-Alexander,0.043790,1
2207,Nikola Jokić,0.012550,0
2202,Kevin Durant,0.004582,1
2198,Giannis Antetokounmpo,0.004391,1
2196,Joel Embiid,0.003615,1
2210,LeBron James,0.003439,1
2197,Luka Dončić,0.001468,1
2211,Trae Young,0.000863,1
2213,Ja Morant,0.000564,1
2214,Anthony Davis,0.000562,1



24-25


,Player,Probability,Rk
2933,Nikola Jokić,0.032770,1
2931,Shai Gilgeous-Alexander,0.008296,0
2932,Giannis Antetokounmpo,0.006341,1
2951,Zion Williamson,0.004599,1
2953,LeBron James,0.003352,1
2939,Kevin Durant,0.002438,1
2957,Trae Young,0.002098,1
2966,Zach LaVine,0.001001,1
2934,Luka Dončić,0.000745,1
2948,Anthony Davis,0.000743,1



25-26


,Player,Probability,Rk
3673,Nikola Jokić,0.010440,1
3674,Giannis Antetokounmpo,0.004327,1
3667,Shai Gilgeous-Alexander,0.004244,0
3671,Kawhi Leonard,0.001154,1
3680,Kevin Durant,0.000959,1
3666,Luka Dončić,0.000886,1
3686,Cade Cunningham,0.000795,1
3688,James Harden,0.000582,1
3668,Anthony Edwards,0.000556,1
3687,James Harden,0.000504,1


In [61]:
# Predict top 5 MVP candidates for each model and season

test_copy = test.copy()

for name, model in trained_models.items():
    proba = model.predict_proba(X_test)
    pos_class_idx = list(model.classes_).index(0)
    test_copy[f"{name} Probability"] = proba[:, pos_class_idx]

for season in sorted(test_copy["season"].unique()):
    print(f"\nSeason {season}")
    for name in trained_models:
        print(f"\n{name} top 5 MVP predictions")
        display(
            test_copy[test_copy["season"] == season]
            .sort_values(f"{name} Probability", ascending=False)
            [["Player", "Rk", f"{name} Probability"]]
            .head(5)
        )


Season 20-21

Logistic Regression top 5 MVP predictions


,Player,Rk,Logistic Regression Probability
0,Stephen Curry,1,0.999993
2,Damian Lillard,1,0.999980
10,Nikola Jokić,0,0.999977
33,Russell Westbrook,1,0.999973
5,Luka Dončić,1,0.999959



Random Forest top 5 MVP predictions


,Player,Rk,Random Forest Probability
10,Nikola Jokić,0,0.088333
24,Julius Randle,1,0.038333
14,Trae Young,1,0.033333
5,Luka Dončić,1,0.025000
2,Damian Lillard,1,0.020000



XGBoost top 5 MVP predictions


...

,Player,Rk,Random Forest Probability
3670,Tyrese Maxey,1,0.111667
3681,Jamal Murray,1,0.105000
3667,Shai Gilgeous-Alexander,0,0.093333
3680,Kevin Durant,1,0.083333
3673,Nikola Jokić,1,0.081667



XGBoost top 5 MVP predictions


,Player,Rk,XGBoost Probability
3673,Nikola Jokić,1,0.010440
3674,Giannis Antetokounmpo,1,0.004327
3667,Shai Gilgeous-Alexander,0,0.004244
3671,Kawhi Leonard,1,0.001154
3680,Kevin Durant,1,0.000959



LightGBM top 5 MVP predictions


,Player,Rk,LightGBM Probability
3667,Shai Gilgeous-Alexander,0,0.977910
3673,Nikola Jokić,1,0.680191
3671,Kawhi Leonard,1,0.039898
3666,Luka Dončić,1,0.023572
3680,Kevin Durant,1,0.006810
